In [2]:
# =====================================
# 📥 EEG Data Loading (Relative Paths)
# =====================================

import os
import glob
import pandas as pd

def load_eeg_data(data_root):
    """
    Load EEG data from anonymized participant folders.

    Args:
        data_root (str): Root directory containing participant folders (e.g., anonymized_data/p1, p2, ...)

    Returns:
        all_data (pd.DataFrame): Combined dataframe containing all EEG data
                                with patient_id and recording_id columns.
        data_dict (dict): Dictionary mapping recording_id -> dataframe.
    """

    channels = [
        "EEG.AF3", "EEG.F7", "EEG.F3", "EEG.FC5", "EEG.T7", "EEG.P7",
        "EEG.O1", "EEG.O2", "EEG.P8", "EEG.T8", "EEG.FC6", "EEG.F4",
        "EEG.F8", "EEG.AF4", "MarkerValueInt"
    ]

    all_data = []
    data_dict = {}

    # Find all CSV files recursively
    file_paths = glob.glob(
        os.path.join(data_root, "**", "*.csv"),
        recursive=True
    )

    for path in file_paths:
        # Extract identifiers
        recording_id = os.path.splitext(os.path.basename(path))[0]
        patient_id = os.path.basename(os.path.dirname(path))  # e.g., p3, p11

        # Load CSV
        df = pd.read_csv(path, skiprows=1, usecols=channels)

        # Add identifiers
        df["patient_id"] = patient_id
        df["recording_id"] = recording_id

        # Store
        data_dict[recording_id] = df
        all_data.append(df)

    all_data = pd.concat(all_data, ignore_index=True)

    print(f"✅ Loaded {len(file_paths)} recordings")
    print(f"🔹 Combined dataset shape: {all_data.shape}")
    print("🔹 Columns:", list(all_data.columns))

    return all_data, data_dict


# =====================================
# 📂 Dataset Root (Relative Path)
# =====================================

DATA_ROOT = "anonymized_data"

all_data, data_dict = load_eeg_data(DATA_ROOT)

✅ Loaded 73 recordings
🔹 Combined dataset shape: (4087555, 17)
🔹 Columns: ['EEG.AF3', 'EEG.F7', 'EEG.F3', 'EEG.FC5', 'EEG.T7', 'EEG.P7', 'EEG.O1', 'EEG.O2', 'EEG.P8', 'EEG.T8', 'EEG.FC6', 'EEG.F4', 'EEG.F8', 'EEG.AF4', 'MarkerValueInt', 'patient_id', 'recording_id']


In [3]:
# =====================================
# 🏷️ EEG Label Mapping Function
# =====================================

def replace_marker_with_label(all_data, data_dict):
    """
    Replace numeric MarkerValueInt values with descriptive MI labels.
    Unlabeled samples are preserved.
    """

    label_map = {
        82: "Foot Dorsiflexion",
        25: "Shoulder Shrug",
        24: "Eyebrow Raise",
        23: "Left Hand Grasp",
        5:  "Knee Extension",
        9:  "Right Hand Grasp",
        7:  "Lip Purse",
        2:  "Jaw Clench",
        1:  "Fail"
    }

    updated_all_data = all_data.copy()
    updated_all_data["Label"] = updated_all_data["MarkerValueInt"].map(label_map)
    updated_all_data.drop(columns=["MarkerValueInt"], inplace=True)

    updated_data_dict = {}
    for name, df in data_dict.items():
        df_copy = df.copy()
        df_copy["Label"] = df_copy["MarkerValueInt"].map(label_map)
        df_copy.drop(columns=["MarkerValueInt"], inplace=True)
        updated_data_dict[name] = df_copy

    print("✅ Replaced numeric markers with descriptive labels (unlabeled samples preserved).")
    return updated_all_data, updated_data_dict

all_data, data_dict = replace_marker_with_label(all_data, data_dict)

✅ Replaced numeric markers with descriptive labels (unlabeled samples preserved).


In [4]:
# =====================================
# 🧩 EEG Window Segmentation Function (Patient-Aware)
# =====================================

import numpy as np
import pandas as pd

def create_timeframes(df, sampling_rate=256, pre_seconds=2, post_seconds=3):
    """
    Split continuous EEG data into windows around labeled events.
    Preserves patient identity per window.
    """

    channels = [col for col in df.columns if col.startswith("EEG.")]
    pre_samples = int(pre_seconds * sampling_rate)
    post_samples = int(post_seconds * sampling_rate)

    X_windows, y_labels, patient_ids = [], [], []

    labeled_indices = df.index[df["Label"].notna()]

    for idx in labeled_indices:
        label = df.at[idx, "Label"]

        if label == "Fail":
            if X_windows:
                X_windows.pop()
                y_labels.pop()
                patient_ids.pop()
            continue

        start_idx = idx - pre_samples
        end_idx = idx + post_samples

        if start_idx < 0 or end_idx >= len(df):
            continue

        segment = df.iloc[start_idx:end_idx][channels].values
        patient_id = df.at[idx, "patient_id"]

        X_windows.append(segment)
        y_labels.append(label)
        patient_ids.append(patient_id)

    return X_windows, y_labels, patient_ids


# =====================================
# 🔁 Create windows for all recordings
# =====================================

X_all, y_all, patient_ids = [], [], []

for _, df_rec in data_dict.items():
    X_s, y_s, pid_s = create_timeframes(df_rec, sampling_rate=256)
    X_all.extend(X_s)
    y_all.extend(y_s)
    patient_ids.extend(pid_s)

print(f"✅ Total windows: {len(X_all)}")
print(f"✅ Unique patients: {len(set(patient_ids))}")

✅ Total windows: 1716
✅ Unique patients: 11


In [9]:
# ==========================================================
# 🔁 LOSO — 0% vs 50% Calibration
#    Filter-Bank Riemann + Tangent Space + SVM
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
RANDOM_STATE = 42
FS = 256.0

bands = [
    (8, 12),   # μ
    (13, 20),  # low-β
    (20, 30),  # high-β
    (4, 7),    # θ
]

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def ensure_nct(X):
    n, a, b = X.shape
    if a <= 64 and b >= 50: return X
    if b <= 64 and a >= 50: return X.transpose(0, 2, 1)
    return X

def bandpass(data, lo, hi, fs=FS, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=2)

def fb_riemann_ts(Xtr, Xte, ytr, bands):
    feats_tr, feats_te = [], []
    for lo, hi in bands:
        Xtr_f = bandpass(Xtr, lo, hi)
        Xte_f = bandpass(Xte, lo, hi)
        cov_tr = Covariances(estimator='oas').fit_transform(Xtr_f)
        cov_te = Covariances(estimator='oas').transform(Xte_f)
        ts = TangentSpace()
        ts.fit(cov_tr, ytr)
        feats_tr.append(ts.transform(cov_tr))
        feats_te.append(ts.transform(cov_te))
    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

def train_eval_svm(X_train, y_train, X_test, y_test):
    """Fit SVM with fixed RBF params and return metrics."""
    clf = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')
    clf.fit(X_train, y_train)

    t0 = time.perf_counter()
    y_pred = clf.predict(X_test)
    t1 = time.perf_counter()

    inf_ms = (t1 - t0) / len(y_pred) * 1000
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro')
    f1   = f1_score(y_test, y_pred, average='macro')
    return acc, prec, rec, f1, inf_ms

# -------------------------------------------------
# Main LOSO loop
# -------------------------------------------------
unique_patients = np.unique(np.asarray(patient_ids))

results_0  = []   # pure LOSO  (0% calibration)
results_50 = []   # LOSO + 50% calibration

print('Running LOSO FB-Riemann + TS + SVM (0% and 50% calibration)\n')
print(f"{'Subject':<12} {'0% Acc':>8} {'0% F1':>8} {'50% Acc':>9} {'50% F1':>8}")
print('-' * 50)

pbar = tqdm(unique_patients, desc='LOSO subjects', unit='subj')

for pid in pbar:
    pbar.set_description(f"Subject {pid}")

    # --------------------------------------------------
    # Isolate held-out subject vs population
    # --------------------------------------------------
    X_np   = np.asarray(X_all, dtype=np.float32)
    y_np   = np.asarray(y_all)
    pid_np = np.asarray(patient_ids)

    mask   = pid_np == pid
    X_pop  = X_np[~mask];  y_pop  = y_np[~mask]
    X_subj = X_np[mask];   y_subj = y_np[mask]

    # --------------------------------------------------
    # Split subject data: 50% cal / 50% test
    # --------------------------------------------------
    X_cal, X_test, y_cal, y_test = train_test_split(
        X_subj, y_subj,
        test_size=0.5,
        stratify=y_subj,
        random_state=RANDOM_STATE
    )

    # --------------------------------------------------
    # Encode labels using the full label set
    # --------------------------------------------------
    le = LabelEncoder()
    le.fit(y_np)
    y_pop_e  = le.transform(y_pop)
    y_cal_e  = le.transform(y_cal)
    y_test_e = le.transform(y_test)

    # --------------------------------------------------
    # Shape + de-mean
    # --------------------------------------------------
    X_pop  = ensure_nct(X_pop);  X_pop  -= X_pop.mean(axis=2,  keepdims=True)
    X_cal  = ensure_nct(X_cal);  X_cal  -= X_cal.mean(axis=2,  keepdims=True)
    X_test = ensure_nct(X_test); X_test -= X_test.mean(axis=2, keepdims=True)

    # ==================================================
    # PROTOCOL A — Pure LOSO (0% calibration)
    # ==================================================
    pbar.set_postfix_str(f"{pid}: features (0%)")
    X_test_full = np.concatenate([X_cal, X_test], axis=0)
    y_test_full = np.concatenate([y_cal_e, y_test_e])

    Xpop_ts_a, Xtest_ts_a = fb_riemann_ts(X_pop, X_test_full, y_pop_e, bands)

    scaler_a = StandardScaler()
    Xpop_ts_a  = scaler_a.fit_transform(Xpop_ts_a)
    Xtest_ts_a = scaler_a.transform(Xtest_ts_a)

    pbar.set_postfix_str(f"{pid}: training SVM (0%)")
    acc_0, prec_0, rec_0, f1_0, inf_0 = train_eval_svm(
        Xpop_ts_a, y_pop_e, Xtest_ts_a, y_test_full
    )
    results_0.append([acc_0, prec_0, rec_0, f1_0, inf_0])

    # ==================================================
    # PROTOCOL B — LOSO + 50% calibration
    # ==================================================
    pbar.set_postfix_str(f"{pid}: features (50%)")
    Xpop_ts_b, Xcal_ts_b = fb_riemann_ts(X_pop, X_cal, y_pop_e, bands)
    _,         Xtest_ts_b = fb_riemann_ts(X_pop, X_test, y_pop_e, bands)

    scaler_b = StandardScaler()
    scaler_b.fit(np.vstack([Xpop_ts_b, Xcal_ts_b]))
    Xpop_ts_b  = scaler_b.transform(Xpop_ts_b)
    Xcal_ts_b  = scaler_b.transform(Xcal_ts_b)
    Xtest_ts_b = scaler_b.transform(Xtest_ts_b)

    X_train_b = np.vstack([Xpop_ts_b, Xcal_ts_b])
    y_train_b = np.concatenate([y_pop_e, y_cal_e])

    pbar.set_postfix_str(f"{pid}: training SVM (50%)")
    acc_50, prec_50, rec_50, f1_50, inf_50 = train_eval_svm(
        X_train_b, y_train_b, Xtest_ts_b, y_test_e
    )
    results_50.append([acc_50, prec_50, rec_50, f1_50, inf_50])

    pbar.set_postfix_str(f"{pid}: done — 0% Acc={acc_0:.3f} | 50% Acc={acc_50:.3f}")
    print(f"{pid:<12} {acc_0:>8.3f} {f1_0:>8.3f} {acc_50:>9.3f} {f1_50:>8.3f}")

# -------------------------------------------------
# Aggregate
# -------------------------------------------------
r0  = np.array(results_0)
r50 = np.array(results_50)

print('\n📊 Summary — FB-Riemann + TS + SVM')
print(f"{'Protocol':<30} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'Inf(ms)':>10}")
print('-' * 76)
print(f"{'Pure LOSO (0% cal)':<30} "
      f"{r0[:,0].mean():>8.3f} {r0[:,1].mean():>8.3f} "
      f"{r0[:,2].mean():>8.3f} {r0[:,3].mean():>8.3f} {r0[:,4].mean():>10.3f}")
print(f"{'':30} +/- {r0[:,0].std():.3f}    +/- {r0[:,3].std():.3f}")
print(f"{'LOSO + 50% Calibration':<30} "
      f"{r50[:,0].mean():>8.3f} {r50[:,1].mean():>8.3f} "
      f"{r50[:,2].mean():>8.3f} {r50[:,3].mean():>8.3f} {r50[:,4].mean():>10.3f}")
print(f"{'':30} +/- {r50[:,0].std():.3f}    +/- {r50[:,3].std():.3f}")
print(f"{'Global Stratified (reference)':<30} {'0.928':>8} {'0.932':>8} {'0.925':>8} {'0.927':>8} {'0.294':>10}")

Running LOSO FB-Riemann + TS + SVM (0% and 50% calibration)

Subject        0% Acc    0% F1   50% Acc   50% F1
--------------------------------------------------


LOSO subjects:   0%|          | 0/11 [00:00<?, ?subj/s]

p1              0.194    0.093     0.898    0.897
p10             0.042    0.013     0.986    0.745
p11             0.000    0.000     0.975    0.966
p2              0.251    0.120     0.943    0.944
p3              0.000    0.000     0.961    0.960
p4              0.159    0.050     0.909    0.902
p5              0.096    0.053     0.966    0.967
p6              0.053    0.042     0.915    0.913
p7              0.000    0.000     0.862    0.857
p8              0.167    0.072     0.962    0.963
p9              0.024    0.024     0.793    0.771

📊 Summary — FB-Riemann + TS + SVM
Protocol                            Acc     Prec      Rec       F1    Inf(ms)
----------------------------------------------------------------------------
Pure LOSO (0% cal)                0.090    0.117    0.048    0.042      0.736
                               +/- 0.085    +/- 0.038
LOSO + 50% Calibration            0.925    0.909    0.899    0.899      0.871
                               +/- 0.055    +/- 0.

In [10]:
# ==========================================================
# 🔁 LOSO — 25% Calibration
#    Filter-Bank Riemann + Tangent Space + SVM and LogReg
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
CALIBRATION_RATIO = 0.25
RANDOM_STATE = 42
FS = 256.0

bands = [
    (8, 12),   # μ
    (13, 20),  # low-β
    (20, 30),  # high-β
    (4, 7),    # θ
]

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def ensure_nct(X):
    n, a, b = X.shape
    if a <= 64 and b >= 50: return X
    if b <= 64 and a >= 50: return X.transpose(0, 2, 1)
    return X

def bandpass(data, lo, hi, fs=FS, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=2)

def fb_riemann_ts(Xtr, Xte, ytr, bands):
    feats_tr, feats_te = [], []
    for lo, hi in bands:
        Xtr_f = bandpass(Xtr, lo, hi)
        Xte_f = bandpass(Xte, lo, hi)
        cov_tr = Covariances(estimator='oas').fit_transform(Xtr_f)
        cov_te = Covariances(estimator='oas').transform(Xte_f)
        ts = TangentSpace()
        ts.fit(cov_tr, ytr)
        feats_tr.append(ts.transform(cov_tr))
        feats_te.append(ts.transform(cov_te))
    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

def train_eval_svm(X_train, y_train, X_test, y_test):
    clf = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')
    clf.fit(X_train, y_train)
    t0 = time.perf_counter()
    y_pred = clf.predict(X_test)
    t1 = time.perf_counter()
    inf_ms = (t1 - t0) / len(y_pred) * 1000
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro')
    f1   = f1_score(y_test, y_pred, average='macro')
    return acc, prec, rec, f1, inf_ms

def train_eval_logreg(X_train, y_train, X_test, y_test):
    clf = LogisticRegression(
        solver='lbfgs',
        max_iter=3000,
        class_weight='balanced',
        n_jobs=1,
        random_state=RANDOM_STATE
    )
    clf.fit(X_train, y_train)
    t0 = time.perf_counter()
    y_pred = clf.predict(X_test)
    t1 = time.perf_counter()
    inf_ms = (t1 - t0) / len(y_pred) * 1000
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro')
    f1   = f1_score(y_test, y_pred, average='macro')
    return acc, prec, rec, f1, inf_ms

# -------------------------------------------------
# Main LOSO loop
# -------------------------------------------------
unique_patients = np.unique(np.asarray(patient_ids))

results_svm = []
results_lr  = []

print(f'Running LOSO FB-Riemann + TS (SVM and LogReg) with {int(CALIBRATION_RATIO*100)}% calibration\n')
print(f"{'Subject':<12} {'SVM Acc':>9} {'SVM F1':>8} {'LR Acc':>8} {'LR F1':>8}")
print('-' * 52)

pbar = tqdm(unique_patients, desc='LOSO subjects', unit='subj')

for pid in pbar:
    pbar.set_description(f"Subject {pid}")

    # --------------------------------------------------
    # Isolate held-out subject vs population
    # --------------------------------------------------
    X_np   = np.asarray(X_all, dtype=np.float32)
    y_np   = np.asarray(y_all)
    pid_np = np.asarray(patient_ids)

    mask   = pid_np == pid
    X_pop  = X_np[~mask];  y_pop  = y_np[~mask]
    X_subj = X_np[mask];   y_subj = y_np[mask]

    # --------------------------------------------------
    # Split: 25% cal / 75% test
    # --------------------------------------------------
    X_cal, X_test, y_cal, y_test = train_test_split(
        X_subj, y_subj,
        test_size=1 - CALIBRATION_RATIO,
        stratify=y_subj,
        random_state=RANDOM_STATE
    )

    # --------------------------------------------------
    # Encode labels using the full label set
    # --------------------------------------------------
    le = LabelEncoder()
    le.fit(y_np)
    y_pop_e  = le.transform(y_pop)
    y_cal_e  = le.transform(y_cal)
    y_test_e = le.transform(y_test)

    # --------------------------------------------------
    # Shape + de-mean
    # --------------------------------------------------
    X_pop  = ensure_nct(X_pop);  X_pop  -= X_pop.mean(axis=2,  keepdims=True)
    X_cal  = ensure_nct(X_cal);  X_cal  -= X_cal.mean(axis=2,  keepdims=True)
    X_test = ensure_nct(X_test); X_test -= X_test.mean(axis=2, keepdims=True)

    # --------------------------------------------------
    # Feature extraction — shared for both classifiers
    # --------------------------------------------------
    pbar.set_postfix_str(f"{pid}: extracting features")
    Xpop_ts, Xcal_ts = fb_riemann_ts(X_pop, X_cal, y_pop_e, bands)
    _,       Xtest_ts = fb_riemann_ts(X_pop, X_test, y_pop_e, bands)

    scaler = StandardScaler()
    scaler.fit(np.vstack([Xpop_ts, Xcal_ts]))
    Xpop_ts  = scaler.transform(Xpop_ts)
    Xcal_ts  = scaler.transform(Xcal_ts)
    Xtest_ts = scaler.transform(Xtest_ts)

    X_train = np.vstack([Xpop_ts, Xcal_ts])
    y_train = np.concatenate([y_pop_e, y_cal_e])

    # --------------------------------------------------
    # SVM
    # --------------------------------------------------
    pbar.set_postfix_str(f"{pid}: training SVM")
    acc_svm, prec_svm, rec_svm, f1_svm, inf_svm = train_eval_svm(
        X_train, y_train, Xtest_ts, y_test_e
    )
    results_svm.append([acc_svm, prec_svm, rec_svm, f1_svm, inf_svm])

    # --------------------------------------------------
    # Logistic Regression
    # --------------------------------------------------
    pbar.set_postfix_str(f"{pid}: training LogReg")
    acc_lr, prec_lr, rec_lr, f1_lr, inf_lr = train_eval_logreg(
        X_train, y_train, Xtest_ts, y_test_e
    )
    results_lr.append([acc_lr, prec_lr, rec_lr, f1_lr, inf_lr])

    pbar.set_postfix_str(f"{pid}: done")
    print(f"{pid:<12} {acc_svm:>9.3f} {f1_svm:>8.3f} {acc_lr:>8.3f} {f1_lr:>8.3f}")

# -------------------------------------------------
# Aggregate
# -------------------------------------------------
rs  = np.array(results_svm)
rl  = np.array(results_lr)

print(f'\n📊 Summary — FB-Riemann + TS ({int(CALIBRATION_RATIO*100)}% Calibration)')
print(f"{'Protocol':<35} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'Inf(ms)':>10}")
print('-' * 81)
print(f"{'LOSO 25% cal + SVM':<35} "
      f"{rs[:,0].mean():>8.3f} {rs[:,1].mean():>8.3f} "
      f"{rs[:,2].mean():>8.3f} {rs[:,3].mean():>8.3f} {rs[:,4].mean():>10.3f}")
print(f"{'':35} +/- {rs[:,0].std():.3f}    +/- {rs[:,3].std():.3f}")
print(f"{'LOSO 25% cal + LogReg':<35} "
      f"{rl[:,0].mean():>8.3f} {rl[:,1].mean():>8.3f} "
      f"{rl[:,2].mean():>8.3f} {rl[:,3].mean():>8.3f} {rl[:,4].mean():>10.3f}")
print(f"{'':35} +/- {rl[:,0].std():.3f}    +/- {rl[:,3].std():.3f}")
print(f"{'LOSO 50% cal + SVM (ref)':<35} {'0.925':>8} {'0.909':>8} {'0.899':>8} {'0.899':>8} {'0.871':>10}")
print(f"{'Global Stratified SVM (ref)':<35} {'0.928':>8} {'0.932':>8} {'0.925':>8} {'0.927':>8} {'0.294':>10}")


Running LOSO FB-Riemann + TS (SVM and LogReg) with 25% calibration

Subject        SVM Acc   SVM F1   LR Acc    LR F1
----------------------------------------------------


LOSO subjects:   0%|          | 0/11 [00:00<?, ?subj/s]

p1               0.612    0.584    0.592    0.383
p10              0.907    0.556    0.769    0.345
p11              0.808    0.351    0.650    0.304
p2               0.871    0.868    0.765    0.441
p3               0.921    0.617    0.776    0.408
p4               0.879    0.871    0.712    0.473
p5               0.887    0.719    0.774    0.467
p6               0.859    0.851    0.676    0.357
p7               0.672    0.531    0.527    0.363
p8               0.872    0.697    0.667    0.458
p9               0.650    0.497    0.439    0.256

📊 Summary — FB-Riemann + TS (25% Calibration)
Protocol                                 Acc     Prec      Rec       F1    Inf(ms)
---------------------------------------------------------------------------------
LOSO 25% cal + SVM                     0.813    0.679    0.647    0.649      0.587
                                    +/- 0.107    +/- 0.161
LOSO 25% cal + LogReg                  0.668    0.413    0.373    0.387      0.004
             

In [11]:
# ==========================================================
# 🔁 LOSO — 50% Calibration
#    Filter-Bank Riemann + Tangent Space + Logistic Regression
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
CALIBRATION_RATIO = 0.50
RANDOM_STATE = 42
FS = 256.0

bands = [
    (8, 12),   # μ
    (13, 20),  # low-β
    (20, 30),  # high-β
    (4, 7),    # θ
]

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def ensure_nct(X):
    n, a, b = X.shape
    if a <= 64 and b >= 50: return X
    if b <= 64 and a >= 50: return X.transpose(0, 2, 1)
    return X

def bandpass(data, lo, hi, fs=FS, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=2)

def fb_riemann_ts(Xtr, Xte, ytr, bands):
    feats_tr, feats_te = [], []
    for lo, hi in bands:
        Xtr_f = bandpass(Xtr, lo, hi)
        Xte_f = bandpass(Xte, lo, hi)
        cov_tr = Covariances(estimator='oas').fit_transform(Xtr_f)
        cov_te = Covariances(estimator='oas').transform(Xte_f)
        ts = TangentSpace()
        ts.fit(cov_tr, ytr)
        feats_tr.append(ts.transform(cov_tr))
        feats_te.append(ts.transform(cov_te))
    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

# -------------------------------------------------
# Main LOSO loop
# -------------------------------------------------
unique_patients = np.unique(np.asarray(patient_ids))
results_lr = []

print(f'Running LOSO FB-Riemann + TS + LogReg with {int(CALIBRATION_RATIO*100)}% calibration\n')
print(f"{'Subject':<12} {'Acc':>8} {'F1':>8}")
print('-' * 32)

pbar = tqdm(unique_patients, desc='LOSO subjects', unit='subj')

for pid in pbar:
    pbar.set_description(f"Subject {pid}")

    X_np   = np.asarray(X_all, dtype=np.float32)
    y_np   = np.asarray(y_all)
    pid_np = np.asarray(patient_ids)

    mask   = pid_np == pid
    X_pop  = X_np[~mask];  y_pop  = y_np[~mask]
    X_subj = X_np[mask];   y_subj = y_np[mask]

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_subj, y_subj,
        test_size=1 - CALIBRATION_RATIO,
        stratify=y_subj,
        random_state=RANDOM_STATE
    )

    le = LabelEncoder()
    le.fit(y_np)
    y_pop_e  = le.transform(y_pop)
    y_cal_e  = le.transform(y_cal)
    y_test_e = le.transform(y_test)

    X_pop  = ensure_nct(X_pop);  X_pop  -= X_pop.mean(axis=2,  keepdims=True)
    X_cal  = ensure_nct(X_cal);  X_cal  -= X_cal.mean(axis=2,  keepdims=True)
    X_test = ensure_nct(X_test); X_test -= X_test.mean(axis=2, keepdims=True)

    pbar.set_postfix_str(f"{pid}: extracting features")
    Xpop_ts, Xcal_ts = fb_riemann_ts(X_pop, X_cal, y_pop_e, bands)
    _,       Xtest_ts = fb_riemann_ts(X_pop, X_test, y_pop_e, bands)

    scaler = StandardScaler()
    scaler.fit(np.vstack([Xpop_ts, Xcal_ts]))
    Xpop_ts  = scaler.transform(Xpop_ts)
    Xcal_ts  = scaler.transform(Xcal_ts)
    Xtest_ts = scaler.transform(Xtest_ts)

    X_train = np.vstack([Xpop_ts, Xcal_ts])
    y_train = np.concatenate([y_pop_e, y_cal_e])

    pbar.set_postfix_str(f"{pid}: training LogReg")
    clf = LogisticRegression(
        solver='lbfgs',
        max_iter=3000,
        class_weight='balanced',
        n_jobs=1,
        random_state=RANDOM_STATE
    )
    clf.fit(X_train, y_train)

    t0 = time.perf_counter()
    y_pred = clf.predict(Xtest_ts)
    t1 = time.perf_counter()

    inf_ms = (t1 - t0) / len(y_pred) * 1000
    acc  = accuracy_score(y_test_e, y_pred)
    prec = precision_score(y_test_e, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test_e, y_pred, average='macro')
    f1   = f1_score(y_test_e, y_pred, average='macro')

    results_lr.append([acc, prec, rec, f1, inf_ms])
    pbar.set_postfix_str(f"{pid}: done")
    print(f"{pid:<12} {acc:>8.3f} {f1:>8.3f}")

# -------------------------------------------------
# Aggregate
# -------------------------------------------------
rl = np.array(results_lr)

print(f'\n📊 LOSO FB-Riemann + TS + LogReg ({int(CALIBRATION_RATIO*100)}% Calibration)')
print(f"{'Protocol':<35} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'Inf(ms)':>10}")
print('-' * 81)
print(f"{'LOSO 50% cal + LogReg':<35} "
      f"{rl[:,0].mean():>8.3f} {rl[:,1].mean():>8.3f} "
      f"{rl[:,2].mean():>8.3f} {rl[:,3].mean():>8.3f} {rl[:,4].mean():>10.3f}")
print(f"{'':35} +/- {rl[:,0].std():.3f}    +/- {rl[:,3].std():.3f}")
print(f"{'LOSO 50% cal + SVM (ref)':<35} {'0.925':>8} {'0.909':>8} {'0.899':>8} {'0.899':>8} {'0.871':>10}")
print(f"{'Global Stratified LR (ref)':<35} {'0.830':>8} {'0.835':>8} {'0.831':>8} {'0.831':>8} {'0.003':>10}")
print(f"{'Global Stratified SVM (ref)':<35} {'0.928':>8} {'0.932':>8} {'0.925':>8} {'0.927':>8} {'0.294':>10}")


Running LOSO FB-Riemann + TS + LogReg with 50% calibration

Subject           Acc       F1
--------------------------------


LOSO subjects:   0%|          | 0/11 [00:00<?, ?subj/s]

p1              0.796    0.637
p10             0.847    0.444
p11             0.812    0.398
p2              0.830    0.665
p3              0.745    0.392
p4              0.830    0.824
p5              0.888    0.602
p6              0.787    0.402
p7              0.678    0.541
p8              0.769    0.524
p9              0.524    0.348

📊 LOSO FB-Riemann + TS + LogReg (50% Calibration)
Protocol                                 Acc     Prec      Rec       F1    Inf(ms)
---------------------------------------------------------------------------------
LOSO 50% cal + LogReg                  0.773    0.542    0.516    0.525      0.008
                                    +/- 0.095    +/- 0.140
LOSO 50% cal + SVM (ref)               0.925    0.909    0.899    0.899      0.871
Global Stratified LR (ref)             0.830    0.835    0.831    0.831      0.003
Global Stratified SVM (ref)            0.928    0.932    0.925    0.927      0.294


In [12]:
# ==========================================================
# 🔁 LOSO — Pure (0% Calibration)
#    Filter-Bank Riemann + Tangent Space + Logistic Regression
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
RANDOM_STATE = 42
FS = 256.0

bands = [
    (8, 12),   # μ
    (13, 20),  # low-β
    (20, 30),  # high-β
    (4, 7),    # θ
]

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def ensure_nct(X):
    n, a, b = X.shape
    if a <= 64 and b >= 50: return X
    if b <= 64 and a >= 50: return X.transpose(0, 2, 1)
    return X

def bandpass(data, lo, hi, fs=FS, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=2)

def fb_riemann_ts(Xtr, Xte, ytr, bands):
    feats_tr, feats_te = [], []
    for lo, hi in bands:
        Xtr_f = bandpass(Xtr, lo, hi)
        Xte_f = bandpass(Xte, lo, hi)
        cov_tr = Covariances(estimator='oas').fit_transform(Xtr_f)
        cov_te = Covariances(estimator='oas').transform(Xte_f)
        ts = TangentSpace()
        ts.fit(cov_tr, ytr)
        feats_tr.append(ts.transform(cov_tr))
        feats_te.append(ts.transform(cov_te))
    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

# -------------------------------------------------
# Main LOSO loop
# -------------------------------------------------
unique_patients = np.unique(np.asarray(patient_ids))
results_lr = []

print('Running LOSO FB-Riemann + TS + LogReg (0% calibration — pure LOSO)\n')
print(f"{'Subject':<12} {'Acc':>8} {'F1':>8}")
print('-' * 32)

pbar = tqdm(unique_patients, desc='LOSO subjects', unit='subj')

for pid in pbar:
    pbar.set_description(f"Subject {pid}")

    X_np   = np.asarray(X_all, dtype=np.float32)
    y_np   = np.asarray(y_all)
    pid_np = np.asarray(patient_ids)

    mask   = pid_np == pid
    X_pop  = X_np[~mask];  y_pop  = y_np[~mask]
    X_test = X_np[mask];   y_test = y_np[mask]

    # Encode using full label set
    le = LabelEncoder()
    le.fit(y_np)
    y_pop_e  = le.transform(y_pop)
    y_test_e = le.transform(y_test)

    X_pop  = ensure_nct(X_pop);  X_pop  -= X_pop.mean(axis=2,  keepdims=True)
    X_test = ensure_nct(X_test); X_test -= X_test.mean(axis=2, keepdims=True)

    pbar.set_postfix_str(f"{pid}: extracting features")
    Xpop_ts, Xtest_ts = fb_riemann_ts(X_pop, X_test, y_pop_e, bands)

    scaler = StandardScaler()
    Xpop_ts  = scaler.fit_transform(Xpop_ts)
    Xtest_ts = scaler.transform(Xtest_ts)

    pbar.set_postfix_str(f"{pid}: training LogReg")
    clf = LogisticRegression(
        solver='lbfgs',
        max_iter=3000,
        class_weight='balanced',
        n_jobs=1,
        random_state=RANDOM_STATE
    )
    clf.fit(Xpop_ts, y_pop_e)

    t0 = time.perf_counter()
    y_pred = clf.predict(Xtest_ts)
    t1 = time.perf_counter()

    inf_ms = (t1 - t0) / len(y_pred) * 1000
    acc  = accuracy_score(y_test_e, y_pred)
    prec = precision_score(y_test_e, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test_e, y_pred, average='macro')
    f1   = f1_score(y_test_e, y_pred, average='macro')

    results_lr.append([acc, prec, rec, f1, inf_ms])
    pbar.set_postfix_str(f"{pid}: done")
    print(f"{pid:<12} {acc:>8.3f} {f1:>8.3f}")

# -------------------------------------------------
# Aggregate
# -------------------------------------------------
rl = np.array(results_lr)

print('\n📊 Pure LOSO (0% Calibration) — FB-Riemann + TS + LogReg')
print(f"{'Protocol':<35} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'Inf(ms)':>10}")
print('-' * 81)
print(f"{'LOSO 0% cal + LogReg':<35} "
      f"{rl[:,0].mean():>8.3f} {rl[:,1].mean():>8.3f} "
      f"{rl[:,2].mean():>8.3f} {rl[:,3].mean():>8.3f} {rl[:,4].mean():>10.3f}")
print(f"{'':35} +/- {rl[:,0].std():.3f}    +/- {rl[:,3].std():.3f}")
print(f"{'LOSO 0% cal + SVM (ref)':<35} {'0.090':>8} {'0.117':>8} {'0.048':>8} {'0.042':>8} {'0.736':>10}")


Running LOSO FB-Riemann + TS + LogReg (0% calibration — pure LOSO)

Subject           Acc       F1
--------------------------------


LOSO subjects:   0%|          | 0/11 [00:00<?, ?subj/s]

p1              0.276    0.139
p10             0.125    0.057
p11             0.025    0.016
p2              0.320    0.167
p3              0.020    0.010
p4              0.307    0.120
p5              0.096    0.082
p6              0.064    0.061
p7              0.023    0.029
p8              0.071    0.060
p9              0.037    0.027

📊 Pure LOSO (0% Calibration) — FB-Riemann + TS + LogReg
Protocol                                 Acc     Prec      Rec       F1    Inf(ms)
---------------------------------------------------------------------------------
LOSO 0% cal + LogReg                   0.124    0.197    0.061    0.070      0.003
                                    +/- 0.113    +/- 0.050
LOSO 0% cal + SVM (ref)                0.090    0.117    0.048    0.042      0.736
